# Qualidade de `pesquisa_saida.csv`

Diagnóstico da pesquisa de saída pelos mesmos cinco critérios usados nos outros arquivos, mais a checagem de privacidade do texto livre:

| Critério | Pergunta |
|---|---|
| **Completude** | Está tudo lá? Respostas vazias ou sem conteúdo, e quem deveria estar no arquivo |
| **Validade** | Categorias, datas e números no domínio esperado? Texto livre que na prática é categoria converte? |
| **Consistência** | O texto concorda com as categorias e com os outros arquivos? |
| **Privacidade** | Há e-mail, telefone, CPF ou @ de terceiros no texto? |
| **Unicidade** | Cada cancelamento tem uma resposta só? |
| **Temporalidade** | As datas estão no período certo, antes do corte, no ritmo da base? |

**Só diagnóstico:** nada é corrigido aqui. Cada achado vai para uma tabela-resumo em `saidas/`, com status
(`ok`, `esperado` ou `atenção`) e tratamento sugerido. Nada em `dados/brutos/` é alterado (hash conferido no início e no fim).
Transformar o texto em variáveis (extração com IA generativa, objetivo 5 do case) é outra etapa e não é feito aqui.

**Decisões já registradas que valem aqui** (`decisoes.md`): contas de teste e cópias de `usuarios.csv` removidas; datas
dos logs em horário de Brasília; nenhuma informação posterior ao corte. As situações das assinaturas vêm das tabelas limpas de assinaturas e eventos (notebooks 05 e 06).

**Privacidade:** texto livre aparece só como contagem e respostas mais frequentes, nunca ao lado do `user_id`.

In [1]:
ARQUIVO = "pesquisa_saida.csv"

import hashlib
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)

ARQUIVOS = [ARQUIVO, "usuarios.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

achados = []

def achado(criterio, checagem, resultado, linhas, tratamento, status=None):
    # status: ok (nada encontrado), esperado (encontrado, mas é comportamento do negócio) ou atenção.
    if status is None:
        status = "ok" if linhas == 0 else "atenção"
    achados.append({"criterio": criterio, "checagem": checagem, "status": status,
                    "resultado": resultado, "linhas_afetadas": int(linhas), "tratamento_sugerido": tratamento})

# respostas sem conteúdo (vazio disfarçado), pelo checklist da skill
SEM_CONTEUDO = {"nada", "ok", "x", ".", "-", "asdf", "kkkk", "sla", "nao sei", "não sei", "nan", "null", "none", "n/a"}

def sem_conteudo(s):
    t = s.str.strip().str.lower()
    so_simbolo = t.ne("") & ~t.str.contains(r"[a-z0-9à-ú]", regex=True)   # só emoji ou pontuação
    return t.isin(SEM_CONTEUDO) | so_simbolo

PII = {
    "e-mail": r"[\w.+-]+@[\w-]+\.[a-z]{2,}",
    "telefone": r"\(?\b\d{2}\)?\s?9?\d{4}-?\d{4}\b",
    "CPF": r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b",
    "@ de perfil": r"(?<![\w.])@\w{3,}",
}

def contar_pii(s):
    return {nome: int(s.str.contains(rx, regex=True, case=False).sum()) for nome, rx in PII.items()}

brutos_usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, keep_default_na=False, usecols=["user_id", "email"])
email = brutos_usuarios.set_index("user_id")["email"].str.strip()
TESTE = set(email.index[email.str.lower().str.endswith("@vitaliza.com.br")])
COPIA = set(email.index[email != email.str.lower()])
usuarios = pd.read_parquet(PROCESSADOS / "usuarios_limpa.parquet", columns=["user_id", "data_cadastro", "canal_aquisicao"]).set_index("user_id")

def contas_invalidas(df, criterio="Completude"):
    n_teste, n_copia = int(df["user_id"].isin(TESTE).sum()), int(df["user_id"].isin(COPIA).sum())
    sem_cad = int((~df["user_id"].isin(email.index)).sum())
    achado(criterio, "linhas de contas de teste, cópias ou sem cadastro",
           f"teste: {n_teste}; cópias: {n_copia}; sem cadastro: {sem_cad}", n_teste + n_copia + sem_cad, "Se houver: excluir")

## 0. Leitura

In [2]:
pes = pd.read_csv(BRUTOS / ARQUIVO, dtype=str, keep_default_na=False)
assert len(pes) == 643  # cabeçalho da seção 11 do dicionário
N = len(pes)
print(f"pesquisa: {N:,} linhas, {pes['user_id'].nunique()} contas -> {list(pes.columns)}")
pes["data"] = pd.to_datetime(pes["respondido_em"], format="%Y-%m-%d", errors="coerce")
assinaturas = pd.read_parquet(PROCESSADOS / "assinaturas_limpa.parquet",
                              columns=["user_id", "canal_cobranca", "status_atual", "data_fim"]).set_index("user_id")
eventos = pd.read_parquet(PROCESSADOS / "eventos_assinatura_limpo.parquet", columns=["user_id", "data_br", "evento"])
cancel_web = eventos[eventos["evento"] == "cancelamento_web"].copy()
cancel_web["data"] = cancel_web["data_br"].dt.normalize()
pes["canal_cobranca"] = pes["user_id"].map(assinaturas["canal_cobranca"])
pes["status_atual"] = pes["user_id"].map(assinaturas["status_atual"])
pes["data_fim"] = pes["user_id"].map(assinaturas["data_fim"])
pes["motivo_principal"].value_counts()

pesquisa: 643 linhas, 641 contas -> ['resposta_id', 'user_id', 'respondido_em', 'motivo_principal', 'comentario']


motivo_principal
preco               264
outro               223
nao_uso              81
outro_app            34
problema_tecnico     27
saude                14
Name: count, dtype: int64

## 1. Completude

O comentário é opcional. A população é quem cancelou pelo site desde 01/06/2026 (evento `cancelamento_web`).
Números-prova: 641 contas; 353 respondentes entre os fins de assinatura de junho a setembro e 289 respostas de quem
cancelou e seguia ativo em 30/09 (dicionário e enunciado).

In [3]:
vazio = int((pes["comentario"].str.strip() == "").sum())
lixo = int(sem_conteudo(pes["comentario"]).sum())
achado("Completude", "comentário vazio ou sem conteúdo", f"{vazio} vazios ({100 * vazio / N:.0f}%) e {lixo} sem conteúdo", vazio + lixo,
       "Nenhum: o comentário é opcional", status="esperado" if lixo == 0 else None)
achado("Completude", "motivo_principal vazio", f"{int((pes['motivo_principal'] == '').sum())}", int((pes["motivo_principal"] == "").sum()), "Nenhum")
contas_invalidas(pes)

In [4]:
fins = assinaturas[(assinaturas["data_fim"] >= "2026-06-01") & (assinaturas["data_fim"] <= CORTE)]
contas_cancel = cancel_web.loc[cancel_web["data"] >= "2026-06-01", "user_id"].unique()
respondentes = set(pes["user_id"])
prova = {
    "contas que responderam": (len(respondentes), 641),
    "respondentes entre os fins de jun-set": (len(respondentes & set(fins.index)), 353),
    "respostas de quem seguia ativo no corte": (int((pes["status_atual"] == "ativa").sum()), 289),
    "respondentes sem cancelamento web": (len(respondentes - set(contas_cancel)), 0),
}
tabela = pd.DataFrame(prova, index=["obtido", "esperado"]).T
tabela["ok"] = tabela["obtido"] == tabela["esperado"]
display(tabela)
print(f"cancelamentos web desde 01/06: {len(cancel_web)} eventos de {len(contas_cancel):,} contas; "
      f"responderam {len(respondentes):,} ({100 * len(respondentes) / len(contas_cancel):.0f}%)")
print(f"fins de assinatura de jun a set: {len(fins):,}, {fins['canal_cobranca'].str.startswith('web').sum():,} pelo site")
achado("Completude", "população da pesquisa (números-prova do dicionário e do enunciado)",
       "; ".join(f"{k}: {o}" for k, (o, _) in prova.items()), int((~tabela["ok"]).sum()), "Nenhum")
cobertura = len(respondentes & set(fins.index))
achado("Completude", "cobertura do motivo declarado",
       f"{cobertura} dos {len(fins):,} fins de assinatura de jun-set ({100 * cobertura / len(fins):.0f}%) têm resposta; "
       f"{len(respondentes)} das {len(contas_cancel):,} contas que cancelaram pelo site ({100 * len(respondentes) / len(contas_cancel):.0f}%) "
       "responderam; quem paga pela loja nunca vê a pesquisa", 0,
       "Não generalizar o motivo declarado para todos os que saem (viés de seleção, como diz o enunciado)", status="atenção")

,obtido,esperado,ok
contas que responderam,641,641,True
respondentes entre os fins de jun-set,353,353,True
respostas de quem seguia ativo no corte,289,289,True
respondentes sem cancelamento web,0,0,True


cancelamentos web desde 01/06: 1172 eventos de 1,164 contas; responderam 641 (55%)
fins de assinatura de jun a set: 3,389, 1,118 pelo site


## 2. Validade

In [5]:
MOTIVOS = {"preco", "nao_uso", "problema_tecnico", "outro_app", "saude", "outro"}
fora = int((~pes["motivo_principal"].isin(MOTIVOS)).sum())
achado("Validade", "motivo_principal fora da lista do dicionário", f"{fora}", fora, "Nenhum")
falha = int(pes["data"].isna().sum())
achado("Validade", "respondido_em que não converte", f"{falha}", falha, "Nenhum")
grafias = pes["comentario"].str.strip()
variantes = grafias[grafias != ""].groupby(grafias.str.lower()).nunique()
n_var = int(pes["comentario"].str.strip().str.lower().isin(variantes[variantes > 1].index).sum())
print("comentários que só diferem por maiúsculas:", variantes[variantes > 1].to_dict())
achado("Validade", "mesmo comentário com maiúsculas diferentes", f"{int((variantes > 1).sum())} comentários em {n_var} respostas",
       n_var, "Comparar comentários em minúsculas")

comentários que só diferem por maiúsculas: {'o gps não funciona no meu celular': 2}


## 3. Consistência

A resposta deve vir no dia de um cancelamento web, e só de quem paga pelo site. O motivo marcado deve concordar com o
comentário. Para comparar, classifico os comentários frequentes em temas por palavras-chave (só para medir a concordância;
a extração de verdade fica para a etapa de IA generativa).

In [6]:
web = pes["canal_cobranca"].isin(["web_cartao", "web_pix"])
achado("Consistência", "resposta de quem não paga pelo site", f"{int((~web).sum())}", int((~web).sum()), "Nenhum")
mesmo_dia = pes.merge(cancel_web[["user_id", "data"]], on=["user_id", "data"], how="left", indicator=True)
sem_cancel_no_dia = int(mesmo_dia.drop_duplicates(["resposta_id"])["_merge"].eq("left_only").sum())
achado("Consistência", "resposta sem cancelamento web no mesmo dia", f"{sem_cancel_no_dia}", sem_cancel_no_dia, "Nenhum")

In [7]:
TEMAS = {   # a ordem importa: vale o primeiro tema que casar
    "objetivo cumprido (prova)": r"prova|objetivo cumprido",
    "outro app": r"strava|outro app|nike|garmin connect|outra plataforma",
    "saúde": r"lesão|lesao|médico|medico|cirurgia|joelho|saúde|saude|grávida|gravida",
    "vida pessoal": r"trabalho|filho|mudei|mudança|viagem|faculdade|casamento",
    "problema técnico": r"bug|gps|trav|erro|atualiza|não funciona|nao funciona",
    "preço": r"caro|preço|preco|dinheiro|grana|desconto|pagar",
    "não uso / perdi a vontade": r"usando|vontade|chata|tempo|mesma coisa|mesmo treino|enjoei|cansei",
}
coment = pes["comentario"].str.lower()
tema = pd.Series(None, index=pes.index, dtype=object)
for nome, rx in TEMAS.items():
    tema = tema.where(tema.notna() | ~coment.str.contains(rx, regex=True), nome)
pes["tema_comentario"] = tema
com_tema = pes["tema_comentario"].notna()
print(f"comentários com tema identificado: {com_tema.sum()} de {(coment.str.strip() != '').sum()}")
display(pd.crosstab(pes.loc[com_tema, "tema_comentario"], pes.loc[com_tema, "motivo_principal"]))
ESPERADO = {"problema técnico": "problema_tecnico", "preço": "preco", "não uso / perdi a vontade": "nao_uso",
            "outro app": "outro_app", "saúde": "saude"}
comparavel = pes["tema_comentario"].isin(ESPERADO)
contradiz = comparavel & (pes["motivo_principal"] != pes["tema_comentario"].map(ESPERADO))
prova_marcada = pes["tema_comentario"].eq("objetivo cumprido (prova)")
print(f"tema com motivo correspondente: {comparavel.sum()}; motivo diferente do tema: {contradiz.sum()} ({100 * contradiz.sum() / comparavel.sum():.0f}%)")
print("'objetivo cumprido (prova)' marcado como:", pes.loc[prova_marcada, "motivo_principal"].value_counts().to_dict())
achado("Consistência", "motivo marcado diferente do tema do comentário",
       f"{contradiz.sum()} de {comparavel.sum()} comentários com tema comparável ({100 * contradiz.sum() / comparavel.sum():.0f}%), "
       f"a maioria marcada como 'preco'; e {prova_marcada.sum()} 'objetivo cumprido / era só até a prova', motivo que não existe "
       f"na lista, marcados como {pes.loc[prova_marcada, 'motivo_principal'].value_counts().to_dict()}",
       int(contradiz.sum() + prova_marcada.sum()),
       "Não usar o motivo marcado sozinho: ler o comentário junto (extração com IA generativa) e criar o motivo "
       "'objetivo cumprido' quando o comentário disser")

comentários com tema identificado: 379 de 460


motivo_principal,nao_uso,outro,outro_app,preco,problema_tecnico,saude
tema_comentario,,,,,,
não uso / perdi a vontade,42,4,11,39,0,0
objetivo cumprido (prova),5,85,3,65,0,0
problema técnico,2,1,3,22,23,0
saúde,0,1,0,3,0,7
vida pessoal,0,38,1,24,0,0


tema com motivo correspondente: 158; motivo diferente do tema: 86 (54%)
'objetivo cumprido (prova)' marcado como: {'outro': 85, 'preco': 65, 'nao_uso': 5, 'outro_app': 3}


## 4. Privacidade

In [8]:
pii = contar_pii(pes["comentario"])
print(pii)
achado("Privacidade", "e-mail, telefone, CPF ou @ no comentário", f"{pii}", sum(pii.values()), "Se houver: mascarar antes de qualquer envio a IA")

{'e-mail': 0, 'telefone': 0, 'CPF': 0, '@ de perfil': 0}


## 5. Unicidade

Duas respostas da mesma conta só são esperadas se a conta cancelou pelo site duas vezes.

In [9]:
achado("Unicidade", "resposta_id repetido", f"{int(pes['resposta_id'].duplicated().sum())}", int(pes["resposta_id"].duplicated().sum()), "Nenhum")
duas = pes[pes["user_id"].duplicated(keep=False)]
cancel_por_conta = cancel_web[cancel_web["data"] >= "2026-06-01"].groupby("user_id").size()
print(f"contas com duas respostas: {duas['user_id'].nunique()}; cancelamentos web dessas contas: "
      f"{cancel_por_conta.reindex(duas['user_id'].unique()).tolist()}; datas diferentes: {duas.groupby('user_id')['data'].nunique().eq(2).all()}")
achado("Unicidade", "conta com duas respostas",
       f"{duas['user_id'].nunique()} contas, cada uma com dois cancelamentos web em datas diferentes (cancelou, religou e cancelou de novo)",
       len(duas), "Nenhum: são dois cancelamentos; para motivo por conta, usar a resposta mais recente", status="esperado")

contas com duas respostas: 2; cancelamentos web dessas contas: [2, 2]; datas diferentes: True


## 6. Temporalidade

In [10]:
print(f"respondido_em de {pes['data'].min().date()} a {pes['data'].max().date()}")
fora_periodo = int(((pes["data"] < pd.Timestamp("2026-06-01")) | (pes["data"] > CORTE)).sum())
achado("Temporalidade", "resposta antes de 01/06/2026 ou depois do corte", f"{fora_periodo}", fora_periodo, "Nenhum")
mensal = pd.DataFrame({"respostas": pes.groupby(pes["data"].dt.to_period("M")).size(),
                       "cancelamentos web": cancel_web.groupby(cancel_web["data"].dt.to_period("M")).size()}).fillna(0).astype(int)
mensal["% que respondeu"] = (100 * mensal["respostas"] / mensal["cancelamentos web"]).round(0)
display(mensal)
achado("Temporalidade", "taxa de resposta por mês", f"de {mensal['% que respondeu'].min():.0f}% a {mensal['% que respondeu'].max():.0f}%",
       0, "Nenhum")

respondido_em de 2026-06-01 a 2026-09-30


,respostas,cancelamentos web,% que respondeu
data,,,
2026-06,210,378,56.0
2026-07,182,323,56.0
2026-08,154,271,57.0
2026-09,97,200,48.0


## Resumo

In [11]:
resumo = pd.DataFrame(achados)
resumo["% do arquivo"] = (100 * resumo["linhas_afetadas"] / N).round(2)
resumo.to_csv(SAIDAS / f"qualidade_{Path(ARQUIVO).stem}.csv", index=False)
print(resumo["status"].value_counts().to_dict())
resumo

{'ok': 11, 'atenção': 3, 'esperado': 2}


,criterio,checagem,status,resultado,linhas_afetadas,tratamento_sugerido,% do arquivo
0,Completude,comentário vazio ou sem conteúdo,esperado,183 vazios (28%) e 0 sem conteúdo,183,Nenhum: o comentário é opcional,28.46
1,Completude,motivo_principal vazio,ok,0,0,Nenhum,0.00
2,Completude,"linhas de contas de teste, cópias ou sem cadastro",ok,teste: 0; cópias: 0; sem cadastro: 0,0,Se houver: excluir,0.00
3,Completude,população da pesquisa (números-prova do dicionário e do enunciado),ok,contas que responderam: 641; respondentes entre os fins de jun-set...,0,Nenhum,0.00
4,Completude,cobertura do motivo declarado,atenção,"353 dos 3,389 fins de assinatura de jun-set (10%) têm resposta; 64...",0,Não generalizar o motivo declarado para todos os que saem (viés de...,0.00
5,Validade,motivo_principal fora da lista do dicionário,ok,0,0,Nenhum,0.00
6,Validade,respondido_em que não converte,ok,0,0,Nenhum,0.00
7,Validade,mesmo comentário com maiúsculas diferentes,atenção,1 comentários em 20 respostas,20,Comparar comentários em minúsculas,3.11
8,Consistência,resposta de quem não paga pelo site,ok,0,0,Nenhum,0.00
9,Consistência,resposta sem cancelamento web no mesmo dia,ok,0,0,Nenhum,0.00


In [12]:
hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/brutos/ inalterado: True
